In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.appName("Kafka Example").getOrCreate()

hconf = spark._jsc.hadoopConfiguration()
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hconf.set("fs.s3a.endpoint", "http://minio:9000")
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.connection.ssl.enabled", "false")
hconf.set("fs.s3a.access.key", "mmix")
hconf.set("fs.s3a.secret.key", "mmixmmix")
hconf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
# hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
# hconf.set("fs.s3a.endpoint", "s3.ap-northeast-2.amazonaws.com")
# hconf.set("fs.s3a.path.style.access", "false")
# hconf.set("fs.s3a.aws.credentials.provider", "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")

In [ ]:
dataframe = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "confluent-kafka-broker-1:9093,confluent-kafka-broker-2:9096,confluent-kafka-broker-3:9099") \
    .option("kafka.security.protocol", "SASL_PLAINTEXT") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config", "org.apache.kafka.common.security.plain.PlainLoginModule ""required username=admin password=admin;") \
    .option("subscribe", "mmix-products-topic") \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

In [ ]:
# dataframe = spark \
#     .readStream \
#     .format("kafka") \
#     .option("kafka.bootstrap.servers", "confluent-kafka-broker-1:9093,confluent-kafka-broker-2:9096,confluent-kafka-broker-3:9098") \
#     .option("subscribe", "mmix-products-topic") \
#     .option("startingOffsets", "earliest") \
#     .option("failOnDataLoss", "false") \
#     .load()

In [ ]:
# q = dataframe \
#     .selectExpr("topic", "CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value") \
#     .writeStream.format("console").option("truncate", "false") \
#     .trigger(processingTime="5 seconds") \
#     .start()
# q.awaitTermination()

In [ ]:
q = dataframe \
    .selectExpr("topic", "CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value", "timestamp") \
    .writeStream \
    .format("parquet") \
    .option("path", "s3a://mmix-prod-dataengineer-datalakehouse/stream/products/") \
    .option("checkpointLocation", "s3a://mmix-prod-dataengineer-datalakehouse/stream/spark-checkpoints/product/") \
    .option("compression", "snappy") \
    .trigger(processingTime="10 seconds") \
    .start()

q.awaitTermination()